In [1]:
import numpy as np
import torch
import torch.nn.functional as F
from sklearn.metrics import roc_auc_score, log_loss, mean_squared_error
from torch.utils.data import DataLoader
from tqdm import tqdm
from itertools import combinations
from sklearn.model_selection import train_test_split

import heapq
from random import randrange
from random import seed as set_seed
import numpy as np
from numba import njit, prange
from pandas.api.types import is_numeric_dtype

import numpy as np
import pandas as pd
import torch.utils.data

from scipy.sparse.linalg import svds

/home/yusupov/.local/lib/python3.10/site-packages/pandas/core/computation/expressions.py:21: UserWarning: Pandas requires version '2.8.4' or newer of 'numexpr' (version '2.8.1' currently installed).
  from pandas.core.computation.check import NUMEXPR_INSTALLED
/home/yusupov/.local/lib/python3.10/site-packages/pandas/core/arrays/masked.py:60: UserWarning: Pandas requires version '1.3.6' or newer of 'bottleneck' (version '1.3.2' currently installed).
  from pandas.core import (


In [2]:
import pandas as pd
import gzip

def parse(path):
    g = gzip.open(path, 'rb')
    for l in g:
        yield eval(l)

def getDF(path):
    i = 0
    df = {}
    for d in parse(path):
        df[i] = d
        i += 1
    return pd.DataFrame.from_dict(df, orient='index')

df = getDF('reviews_Clothing_Shoes_and_Jewelry_5.json.gz')

In [3]:
df.head(10)

,reviewerID,asin,reviewerName,helpful,reviewText,overall,summary,unixReviewTime,reviewTime
0,A1KLRMWW2FWPL4,0000031887,"Amazon Customer ""cameramom""","[0, 0]",This is a great tutu and at a really great pri...,5.0,Great tutu- not cheaply made,1297468800,"02 12, 2011"
1,A2G5TCU2WDFZ65,0000031887,Amazon Customer,"[0, 0]",I bought this for my 4 yr old daughter for dan...,5.0,Very Cute!!,1358553600,"01 19, 2013"
2,A1RLQXYNCMWRWN,0000031887,Carola,"[0, 0]",What can I say... my daughters have it in oran...,5.0,I have buy more than one,1357257600,"01 4, 2013"
3,A8U3FAMSJVHS5,0000031887,Caromcg,"[0, 0]","We bought several tutus at once, and they are ...",5.0,"Adorable, Sturdy",1398556800,"04 27, 2014"
4,A3GEOILWLK86XM,0000031887,CJ,"[0, 0]",Thank you Halo Heaven great product for Little...,5.0,Grammy's Angels Love it,1394841600,"03 15, 2014"
5,A27UF1MSF3DB2,0000031887,"C-Lo ""Cynthia""","[0, 0]",I received this today and I'm not a fan of it ...,4.0,It's ok,1396224000,"03 31, 2014"
6,A16GFPNVF4Y816,0000031887,design maven,"[0, 0]",Bought this as a backup to the regular ballet ...,5.0,Great for dress-up and for ballet practice,1399075200,"05 3, 2014"
7,A2M2APVYIB2U6K,0000031887,Jamie P.,"[0, 0]",Great tutu for a great price. It isn't a &#34;...,5.0,Great value,1356220800,"12 23, 2012"
8,A1NJ71X3YPQNQ9,0000031887,JBerger,"[0, 0]","My daughter liked this, and it with her costum...",4.0,Good,1384041600,"11 10, 2013"
9,A3EERSWHAI6SO,0000031887,"Jeffrey Hollingshead ""Jillian hollingshead""","[7, 8]",For what I paid for two tutus is unbeatable an...,5.0,WOW !! ..is all I have to say!,1349568000,"10 7, 2012"


In [4]:
new_df = df[['reviewerID', 'asin', 'overall', 'unixReviewTime']].copy()
new_df.columns = ['user_id', 'item_id', 'rating', 'timestamp']
new_df['rating'] = 1
new_df.head(5)

,user_id,item_id,rating,timestamp
0,A1KLRMWW2FWPL4,0000031887,1,1297468800
1,A2G5TCU2WDFZ65,0000031887,1,1358553600
2,A1RLQXYNCMWRWN,0000031887,1,1357257600
3,A8U3FAMSJVHS5,0000031887,1,1398556800
4,A3GEOILWLK86XM,0000031887,1,1394841600


In [5]:
new_df['user_id'], unique_user_ids = pd.factorize(new_df['user_id'])

new_df['item_id'], unique_item_ids = pd.factorize(new_df['item_id'])
new_df['user_id'] += 1
new_df['item_id'] += 1
new_df.head(5)

,user_id,item_id,rating,timestamp
0,1,1,1,1297468800
1,2,1,1,1358553600
2,3,1,1,1357257600
3,4,1,1,1398556800
4,5,1,1,1394841600


In [6]:
df_sorted = new_df.sort_values(by='timestamp')

test_treshold = int(len(df_sorted) * 0.98)
val_treshold = int(len(df_sorted) * 0.96)

train_val = df_sorted.head(test_treshold)
warm_test = df_sorted.tail(len(df_sorted) - test_treshold)
test = warm_test.loc[warm_test.groupby('user_id')['timestamp'].idxmax()]
warm_t = warm_test[~warm_test.index.isin(test.index)]


train = df_sorted.head(val_treshold)
test_val = df_sorted.tail(len(df_sorted) - val_treshold)
warm_val = test_val[~test_val.index.isin(warm_test.index)]
val = warm_val.loc[warm_val.groupby('user_id')['timestamp'].idxmax()]
warm_v = warm_val[~warm_val.index.isin(val.index)]




In [7]:
print("Train len: ", len(train))
print("Train users: ", len(train["user_id"].unique()))
print("Val len: ", len(val))
print("Val users: ", len(val["user_id"].unique()))
print("Test len: ", len(test))
print("Test users: ", len(test["user_id"].unique()))
print("Warm val len: ", len(warm_v))
print("Warm val users: ", len(warm_v["user_id"].unique()))
print("Warm test len: ", len(warm_t))
print("Warm test users: ", len(warm_t["user_id"].unique()))
print("test_val intersection users: ", np.intersect1d(val["user_id"].unique(), test["user_id"].unique()).shape[0])

Train len:  267529
Train users:  39038
Val len:  2689
Val users:  2689
Test len:  2794
Test users:  2794
Warm val len:  2885
Warm val users:  1185
Warm test len:  2780
Warm test users:  1093
test_val intersection users:  398


In [8]:
user_items = train.groupby('user_id')['item_id'].apply(list).to_dict()

with open('data2/Clothes/train.txt', 'w') as f:
    for user_id, items in user_items.items():
        f.write(f"{user_id} {' '.join(map(str, items))}\n")

In [9]:
train_warm_v = pd.concat([train, warm_v], ignore_index=True)
user_items = warm_v.groupby('user_id')['item_id'].apply(list).to_dict()

with open('data2/Clothes/warm_val.txt', 'w') as f:
    for user_id, items in user_items.items():
        f.write(f"{user_id} {' '.join(map(str, items))}\n")

In [10]:
filtered_train = train_warm_v[train_warm_v['user_id'].isin(val['user_id'])]
train_val = pd.concat([filtered_train, val], ignore_index=True)

user_items = train_val.groupby('user_id')['item_id'].apply(list).to_dict()
with open('data2/Clothes/valid.txt', 'w') as f:
    for user_id, items in user_items.items():
        if len(items) > 1:
            f.write(f"{user_id} {' '.join(map(str, items))}\n")


In [11]:
unique_indices = val["user_id"].unique()
with open('data2/Clothes/val_users.txt', 'w') as f:
    for index in unique_indices:
        f.write(f"{index}\n")

# Чтение чисел из файла и сохранение их в список
with open('data2/Clothes/val_users.txt', 'r') as f:
    index_list = [int(line.strip()) for line in f]

In [12]:
filtered_train = df_sorted[df_sorted['user_id'].isin(test['user_id'])]


user_items = filtered_train.groupby('user_id')['item_id'].apply(list).to_dict()
with open('data2/Clothes/test.txt', 'w') as f:
    for user_id, items in user_items.items():
        f.write(f"{user_id} {' '.join(map(str, items))}\n")


In [13]:
user_items = df_sorted.groupby('user_id')['item_id'].apply(list).to_dict()

with open('data2/Clothes/all_data.txt', 'w') as f:
    for user_id, items in user_items.items():
        f.write(f"{user_id} {' '.join(map(str, items))}\n")